# Inferencing — User Guide

Two kinds of reasoning, in very different states: RDFS subclass reasoning is real, and `sh:targetClass`/`sh:class`/`sh:rootClass` already use it — this guide covers what's automatic and the one part that's opt-in. OWL reasoning is not implemented at all — this guide says so plainly rather than leaving it to be discovered by a missing feature.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells top to bottom.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")

## 1. RDFS subclass reasoning: automatic for target/class matching

SHACL's own spec bakes `rdfs:subClassOf` reasoning into `sh:targetClass` matching — a shape targeting `ex:Animal` reaches an instance of `ex:Lion` if `ex:Lion rdfs:subClassOf ex:Animal` is asserted in the *data* graph, with no extra flag needed. This isn't a starshacl addition; it's how SHACL's target computation is specified to work everywhere.

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Lion rdfs:subClassOf ex:Animal .
    ex:leo a ex:Lion ; ex:name "Leo" .
    ex:mystery a ex:Lion .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:AnimalShape a sh:NodeShape ;
      sh:targetClass ex:Animal ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

# ex:leo and ex:mystery are both only ever typed ex:Lion, never ex:Animal directly -
# sh:targetClass ex:Animal still reaches both via the subclass relationship
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms)
print("ex:mystery flagged (a ex:Lion, missing ex:name):", "ex:mystery" in result.report_text)
print("ex:leo flagged (a ex:Lion, has ex:name):", "Focus Node: ex:leo" in result.report_text)

conforms: False
ex:mystery flagged (a ex:Lion, missing ex:name): True
ex:leo flagged (a ex:Lion, has ex:name): False


## 2. Opt-in: reading `rdfs:subClassOf` from the shapes graph too

SHACL 1.2 Core's "SHACL Type" definition (Issue 185) notes implementations MAY also consult `rdfs:subClassOf` triples living in the *shapes* graph, not just the data graph — useful when a class hierarchy is defined alongside the shapes rather than shipped with the data. starshacl exposes this as an opt-in `validate()` keyword, `rdfs_subclass_reasoning_includes_shapes_graph`, consulted by the native `sh:rootClass` and list-valued `sh:class` passes. Off by default (matching plain pySHACL's own behavior, which never looks at the shapes graph for this).

In [3]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

    # This rdfs:subClassOf assertion lives only in the SHAPES graph, not the data graph.
    ex:Lion rdfs:subClassOf ex:Animal .

    ex:S a sh:NodeShape ;
      sh:targetNode ex:Zoo ;
      sh:property [
        sh:path ex:holds ;
        sh:rootClass ex:Animal ;
      ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:Zoo ex:holds ex:Lion .
""", format="turtle")

result_default = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms by default (shapes-graph-only subclass assertion ignored):", result_default.conforms)

result_opt_in = StarShaclValidator().validate(
    data_graph=data, shacl_graph=shapes, meta_shacl=False,
    rdfs_subclass_reasoning_includes_shapes_graph=True,
)
print("conforms with rdfs_subclass_reasoning_includes_shapes_graph=True:", result_opt_in.conforms)

conforms by default (shapes-graph-only subclass assertion ignored): False
conforms with rdfs_subclass_reasoning_includes_shapes_graph=True: True


### Full RDFS closure via `inference=`

`validate()` also forwards an `inference=` keyword straight through to pySHACL itself (`"rdfs"`, `"owlrl"`, or `"both"`), which pre-materializes RDFS/OWL entailments over the data graph before validation runs — plain pySHACL passthrough, not a starshacl-specific addition. Useful when a constraint needs to see inferred triples beyond subclass-aware target matching (which, as section 1 showed, already works without it).

## 3. OWL reasoning: not implemented

There is no OWL reasoning anywhere in this stack beyond what plain pySHACL's own `inference="owlrl"`/`"both"` passthrough provides (section 2 above) — no OWL-aware constraint components, no `owl:equivalentClass`/`owl:sameAs`-aware SHACL semantics, nothing starshacl-specific. This is a real, open gap, tracked in the repo-root `Task List` (item 3, "bring in OWL and OWL reasoning into the stack").